# Robust Machine Translation under Noisy Conditions (EN -> VI)

**Approach:** a tiny Transformer **encoder-decoder** (~2.4M trainable
parameters, under both the 5,000,000 hard budget and the 2,500,000 bonus
threshold), trained on a denoised + jointly-BPE-tokenized corpus, with a
generate-many-then-rerank inference pipeline (beam search + low-temperature
sampling, filtered and reranked with a translation-oriented score).

This notebook runs the **entire pipeline end to end**:

1. Setup & data loading
2. Data-centric denoising (self-correct, then delete unrecoverable noise)
3. Baseline: vanilla RNN Seq2Seq (from the exercise notebook) for comparison
4. Tokenizer: joint BPE vocabulary shared across EN/VI
5. Model: tiny Transformer encoder-decoder
6. Training
7. Evaluation: BLEU for greedy / beam search / generate-and-rerank decoding
8. Qualitative analysis: cross-attention visualization + error analysis

Set `DATA_DIR` below to point at the folder containing
`train_noisy.en.txt`, `train.vi.txt`, `val_noisy.en.txt`, `val.vi.txt`,
`test_noisy.en.txt`, `test.vi.txt`.

## 0. Setup

In [ ]:
# If running in a fresh environment (e.g. Colab), uncomment to install deps:
# !pip install -q torch tokenizers wordsegment nltk tqdm matplotlib numpy

import os
import re
import html
import json
import math
import random
import time
from collections import Counter, namedtuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import nltk
try:
    from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
except LookupError:
    nltk.download("punkt")
    from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.normalizers import NFKC, Lowercase, Sequence as NormSequence

from wordsegment import load as _ws_load, segment as _ws_segment, UNIGRAMS as _UNIGRAMS
_ws_load()

SEED = 42
def fix_random_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
fix_random_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ---- Paths (edit DATA_DIR to point at your corpus) ----
DATA_DIR = "./en-vi-translation-data"
OUTPUT_DIR = "./output"
CLEAN_DIR = os.path.join(OUTPUT_DIR, "clean_data")
TOK_DIR = os.path.join(OUTPUT_DIR, "tokenizers")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---- Run-size knobs: keep these small for a quick smoke test, raise them
# for the final report run. ----
MAX_TRAIN_SAMPLES = 30000
EPOCHS = 20
BASELINE_EPOCHS = 5     # the vanilla-RNN baseline is slow per-epoch; keep short
BATCH_SIZE = 64
MAX_LEN = 128

assert os.path.exists(DATA_DIR), (
    f"DATA_DIR={DATA_DIR!r} not found. Point it at the folder containing "
    f"train_noisy.en.txt / train.vi.txt / val_noisy.en.txt / val.vi.txt / "
    f"test_noisy.en.txt / test.vi.txt before running the rest of this notebook."
)
print("Found data directory:", DATA_DIR)

## 1. Data-centric denoising

Two layers of cleanup, applied to the noisy **English source only** (the
Vietnamese target is already clean):

1. **`clean_text`** (conservative, "self-correct"): decode HTML entities,
   collapse elongated characters (`"soooo"` -> `"soo"`) and repeated
   punctuation (`"!!!"` -> `"!"`), strip disallowed characters, and repair
   missing-space concatenations (`"ihavean apple"` -> `"i have an apple"`)
   via dictionary-checked word segmentation (using word-frequency
   statistics, not just membership, to avoid "fixing" gibberish into
   fabricated words).
2. **`super_clean_text`** (aggressive, "self-correct, then delete"): runs
   `clean_text`, then **deletes** any token that still isn't a recognizable
   word afterward -- i.e. removes genuine injected gibberish that couldn't
   be corrected, instead of leaving it in the sentence for the model to
   absorb. Used to build a fully denoised copy of the corpus
   (`preprocess`, below).

In [ ]:
PAD, UNK, SOS, EOS, SEP = "<pad>", "<unk>", "<sos>", "<eos>", "<sep>"
# Direction tags: training on BOTH <toVI> (EN->VI, the real task) and
# <toEN> (VI->EN) examples with the same shared vocab/stack turns the one
# model bidirectional at zero extra inference-time parameters -- lets the
# SAME model later score a candidate translation's reverse probability
# log P(x|y) for reranking, instead of needing a second trained model.
TOEN, TOVI = "<toen>", "<tovi>"
SPECIAL_TOKENS = [PAD, UNK, SOS, EOS, SEP, TOEN, TOVI]
PAD_ID, UNK_ID, SOS_ID, EOS_ID, SEP_ID, TOEN_ID, TOVI_ID = 0, 1, 2, 3, 4, 5, 6

_REPEAT_CHAR_RE = re.compile(r"(.)\1{2,}")
_MULTI_PUNCT_RE = re.compile(r"([!?.,])\1+")
_NON_ALLOWED_RE = re.compile(
    r"[^a-zA-Z0-9.!?'áàảãạâấầẩẫậăắằẳẵặéèẻẽẹêếềểễệíìỉĩịóòỏõọôốồổỗộơớờởỡợúùủũụưứừửữựýỳỷỹỵđ\s]+"
)
_SPACE_RE = re.compile(r"\s+")
_PUNCT_SPACE_RE = re.compile(r"([.!?])")
_ALPHA_RE = re.compile(r"^[a-z]+$")

# wordsegment's UNIGRAMS table is itself noisy (OCR fragments, accidentally
# merged word pairs with nonzero counts, e.g. "weare": 191,043 -- far below
# real-word frequencies like "coworker": 343,002). A minimum-frequency
# cutoff (rather than bare membership) avoids both (a) wrongly treating
# noisy merges as "already a known word" and (b) accepting gibberish
# fragments as valid split pieces.
_MIN_WORD_FREQ = 300_000

def _is_known_word(tok: str) -> bool:
    return _UNIGRAMS.get(tok, 0) >= _MIN_WORD_FREQ

def _fix_concatenation(token: str) -> list:
    """Repair a suspected 'missing space' merge, e.g. 'ihavean' -> ['i','have','an'].
    Only fires when the token isn't already a known word and segmentation
    finds >=2 pieces that are ALL themselves known (high-frequency) words --
    this protects genuine gibberish like 'gtuql' from being "fixed" into a
    fabricated split."""
    if not _ALPHA_RE.match(token) or len(token) < 4 or _is_known_word(token):
        return [token]
    parts = _ws_segment(token)
    if len(parts) >= 2 and all(_is_known_word(p) for p in parts):
        return parts
    return [token]

def clean_text(s: str, stats: dict = None) -> str:
    orig_len = len(s)
    s = html.unescape(s)
    s = s.lower().strip()
    if stats is not None:
        stats["elongation_collapses"] += len(_REPEAT_CHAR_RE.findall(s))
        stats["punct_run_collapses"] += len(_MULTI_PUNCT_RE.findall(s))
    s = _REPEAT_CHAR_RE.sub(r"\1\1", s)
    s = _MULTI_PUNCT_RE.sub(r"\1", s)
    s = _PUNCT_SPACE_RE.sub(r" \1", s)
    if stats is not None:
        stats["disallowed_chars_stripped"] += len(_NON_ALLOWED_RE.findall(s))
    s = _NON_ALLOWED_RE.sub("", s)
    s = _SPACE_RE.sub(" ", s).strip()
    fixed_tokens, n_concat_fixes = [], 0
    for tok in s.split(" "):
        pieces = _fix_concatenation(tok)
        if len(pieces) > 1:
            n_concat_fixes += 1
        fixed_tokens.extend(pieces)
    if stats is not None:
        stats["concatenation_fixes"] += n_concat_fixes
        stats["lines"] += 1
        stats["chars_before"] += orig_len
        stats["chars_after"] += len(" ".join(fixed_tokens))
    return " ".join(fixed_tokens)

def _is_unrecoverable_garbage(tok: str) -> bool:
    if not _ALPHA_RE.match(tok):
        return False
    if len(tok) <= 2:
        return False
    return not _is_known_word(tok)

def super_clean_text(s: str, stats: dict = None) -> str:
    cleaned = clean_text(s, stats=stats)
    kept, n_deleted = [], 0
    for tok in cleaned.split(" "):
        if not tok:
            continue
        if _is_unrecoverable_garbage(tok):
            n_deleted += 1
            continue
        kept.append(tok)
    if stats is not None:
        stats["garbage_tokens_deleted"] = stats.get("garbage_tokens_deleted", 0) + n_deleted
    return _SPACE_RE.sub(" ", " ".join(kept)).strip()

def denoise_report(raw_lines) -> dict:
    stats = {"lines": 0, "elongation_collapses": 0, "punct_run_collapses": 0,
              "disallowed_chars_stripped": 0, "concatenation_fixes": 0,
              "chars_before": 0, "chars_after": 0}
    raw_vocab, cleaned_vocab = set(), set()
    for line in raw_lines:
        raw_vocab.update(line.lower().split())
        cleaned = clean_text(line, stats=stats)
        cleaned_vocab.update(cleaned.split())
    stats["raw_whitespace_vocab_size"] = len(raw_vocab)
    stats["cleaned_whitespace_vocab_size"] = len(cleaned_vocab)
    stats["vocab_reduction_pct"] = 100.0 * (1 - len(cleaned_vocab) / max(1, len(raw_vocab)))
    return stats

# quick smoke test
sample = "I  havean   apple!!!  soooo goood lol vacx blah"
print("raw        :", sample)
print("clean      :", clean_text(sample))
print("super_clean:", super_clean_text(sample))

### 1b. Source-noise augmentation (applied at TRAIN time only)

Beyond what the noise injector already put in `train_noisy.en.txt`,
randomly corrupting the source further at train time (keyboard-adjacent
typos, dropped characters, random casing) exposes the model to a wider
noise distribution than any single fixed file can -- usually matters more
for robustness than the choice of decoding algorithm.

In [ ]:
_QWERTY_NEIGHBORS = {
    "q": "wa", "w": "qes", "e": "wrd", "r": "etf", "t": "ryg", "y": "tuh",
    "u": "yij", "i": "uok", "o": "ipl", "p": "ol", "a": "qsz", "s": "awd",
    "d": "sfe", "f": "dgr", "g": "fht", "h": "gjy", "j": "hku", "k": "jli",
    "l": "ko", "z": "ax", "x": "zc", "c": "xv", "v": "cb", "b": "vn", "n": "bm", "m": "n",
}

def augment_noise(s: str, rng, p_char: float = 0.03, p_delete: float = 0.01, p_case: float = 0.05) -> str:
    out = []
    for ch in s:
        if ch.isalpha() and rng.random() < p_delete:
            continue
        if ch.lower() in _QWERTY_NEIGHBORS and rng.random() < p_char:
            ch = rng.choice(_QWERTY_NEIGHBORS[ch.lower()])
        elif ch.isalpha() and rng.random() < p_case:
            ch = ch.upper() if ch.islower() else ch.lower()
        out.append(ch)
    return "".join(out)

print(augment_noise("i have an apple", random.Random(0)))

## 2. Run the denoising pipeline over the whole corpus (`preprocess`)

Writes a fully denoised copy of the corpus to `output/clean_data/` (English
source: self-corrected + garbage-deleted; Vietnamese target: lightly
normalized only), plus a JSON report of how much noise was fixed/deleted
per split -- quantitative evidence for the report's data-centric analysis.

In [ ]:
SPLITS = [
    ("train_noisy.en.txt", "train.vi.txt", "train"),
    ("val_noisy.en.txt", "val.vi.txt", "val"),
    ("test_noisy.en.txt", "test.vi.txt", "test"),
]
EMPTY_STATS = {"lines": 0, "elongation_collapses": 0, "punct_run_collapses": 0,
                "disallowed_chars_stripped": 0, "concatenation_fixes": 0,
                "garbage_tokens_deleted": 0, "chars_before": 0, "chars_after": 0}

def process_split(src_path, trg_path, out_src_path, out_trg_path):
    with open(src_path, "r", encoding="utf-8") as f:
        src_lines = [l.strip() for l in f.readlines()]
    with open(trg_path, "r", encoding="utf-8") as f:
        trg_lines = [l.strip() for l in f.readlines()]
    assert len(src_lines) == len(trg_lines)

    stats = dict(EMPTY_STATS)
    src_vocab_before, src_vocab_after = set(), set()
    cleaned_src, cleaned_trg, dropped_empty = [], [], 0
    for s, t in zip(src_lines, trg_lines):
        src_vocab_before.update(s.lower().split())
        cs = super_clean_text(s, stats=stats)
        src_vocab_after.update(cs.split())
        ct = clean_text(t)
        if not cs.strip() or not ct.strip():
            dropped_empty += 1
            continue
        cleaned_src.append(cs)
        cleaned_trg.append(ct)

    stats["pairs_in"] = len(src_lines)
    stats["pairs_out"] = len(cleaned_src)
    stats["pairs_dropped_empty"] = dropped_empty
    stats["src_raw_whitespace_vocab_size"] = len(src_vocab_before)
    stats["src_cleaned_whitespace_vocab_size"] = len(src_vocab_after)
    stats["src_vocab_reduction_pct"] = 100.0 * (1 - len(src_vocab_after) / max(1, len(src_vocab_before)))

    os.makedirs(os.path.dirname(out_src_path) or ".", exist_ok=True)
    with open(out_src_path, "w", encoding="utf-8") as f:
        f.write("\n".join(cleaned_src) + "\n")
    with open(out_trg_path, "w", encoding="utf-8") as f:
        f.write("\n".join(cleaned_trg) + "\n")
    return stats

all_stats = {}
for src_name, trg_name, split in SPLITS:
    src_path, trg_path = os.path.join(DATA_DIR, src_name), os.path.join(DATA_DIR, trg_name)
    if not os.path.exists(src_path) or not os.path.exists(trg_path):
        print(f"[{split}] skipped: {src_path} or {trg_path} not found")
        continue
    out_src = os.path.join(CLEAN_DIR, f"{split}_clean.en.txt")
    out_trg = os.path.join(CLEAN_DIR, f"{split}_clean.vi.txt")
    stats = process_split(src_path, trg_path, out_src, out_trg)
    all_stats[split] = stats
    print(f"[{split}] pairs {stats['pairs_in']} -> {stats['pairs_out']} "
          f"(dropped {stats['pairs_dropped_empty']} now-empty), "
          f"garbage tokens deleted: {stats['garbage_tokens_deleted']}, "
          f"src vocab reduction: {stats['src_vocab_reduction_pct']:.1f}%")

preprocess_stats_path = os.path.join(OUTPUT_DIR, "preprocess_stats.json")
with open(preprocess_stats_path, "w", encoding="utf-8") as f:
    json.dump(all_stats, f, indent=2)
print(f"\nSaved stats to {preprocess_stats_path}")

## 3. Baseline: vanilla RNN Seq2Seq (from the exercise notebook)

A minimal, unmodified-style vanilla-RNN encoder/decoder with a
whitespace-token `Vocabulary` (one entry per raw token -- this is what
explodes parameter count on noisy text), greedy decoding only. Trained
briefly (`BASELINE_EPOCHS`) on the **raw noisy** data so its BLEU score is
directly comparable to what the exercise's own baseline would report.
This gives the "improvement over baseline" numbers required by the report.

In [ ]:
def normalize_string(s):
    s = s.lower().strip()
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z0-9.!?áàảãạâấầẩẫậăắằẳẵặéèẻẽẹêếềểễệíìỉĩịóòỏõọôốồổỗộơớờởỡợúùủũụưứừửữựýỳỷỹỵđ\s]+", r"", s)
    s = re.sub(r"\s+", r" ", s)
    return s.strip()

class Vocabulary:
    def __init__(self):
        self.word2index = {"<pad>": 0, "<unk>": 1, "<sos>": 2, "<eos>": 3}
        self.index2word = {0: "<pad>", 1: "<unk>", 2: "<sos>", 3: "<eos>"}
        self.num_words = 4

    def add_sentence(self, sentence):
        for word in sentence.split():
            if word not in self.word2index:
                self.word2index[word] = self.num_words
                self.index2word[self.num_words] = word
                self.num_words += 1

class BaselineDataset(Dataset):
    def __init__(self, pairs, src_vocab, trg_vocab, max_len=50):
        self.pairs, self.src_vocab, self.trg_vocab, self.max_len = pairs, src_vocab, trg_vocab, max_len

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        src_s, trg_s = self.pairs[idx]
        src_tokens = [self.src_vocab.word2index.get(w, 1) for w in src_s.split()][: self.max_len - 2]
        trg_tokens = [self.trg_vocab.word2index.get(w, 1) for w in trg_s.split()][: self.max_len - 2]
        return [2] + src_tokens + [3], [2] + trg_tokens + [3]

def baseline_collate_fn(batch):
    src_list = [torch.tensor(s) for s, _ in batch]
    trg_list = [torch.tensor(t) for _, t in batch]
    src_padded = nn.utils.rnn.pad_sequence(src_list, batch_first=True, padding_value=0)
    trg_padded = nn.utils.rnn.pad_sequence(trg_list, batch_first=True, padding_value=0)
    return src_padded, trg_padded

class BaselineEncoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.RNN(emb_dim, hidden_size, batch_first=True)
    def forward(self, src):
        outputs, hidden = self.rnn(self.embedding(src))
        return outputs, hidden

class BaselineDecoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_size):
        super().__init__()
        self.output_dim = output_dim
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.RNN(emb_dim, hidden_size, batch_first=True)
        self.fc_out = nn.Linear(hidden_size, output_dim)
    def forward(self, input, hidden):
        embedded = self.embedding(input.unsqueeze(1))
        output, hidden = self.rnn(embedded, hidden)
        return self.fc_out(output.squeeze(1)), hidden

class BaselineSeq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder, self.decoder, self.device = encoder, decoder, device
    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size, trg_len = trg.shape
        trg_vocab_size = self.decoder.output_dim
        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size, device=self.device)
        _, hidden = self.encoder(src)
        input = trg[:, 0]
        for t in range(1, trg_len):
            output, hidden = self.decoder(input, hidden)
            outputs[:, t] = output
            teacher_force = random.random() < teacher_forcing_ratio
            input = trg[:, t] if teacher_force else output.argmax(1)
        return outputs

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def read_langs(src_path, trg_path, max_samples=None):
    with open(src_path, "r", encoding="utf-8") as f:
        src_lines = f.readlines()
    with open(trg_path, "r", encoding="utf-8") as f:
        trg_lines = f.readlines()
    pairs = [(normalize_string(s), normalize_string(t)) for s, t in zip(src_lines, trg_lines)]
    return pairs[:max_samples] if max_samples else pairs

train_src, train_trg = os.path.join(DATA_DIR, "train_noisy.en.txt"), os.path.join(DATA_DIR, "train.vi.txt")
val_src, val_trg = os.path.join(DATA_DIR, "val_noisy.en.txt"), os.path.join(DATA_DIR, "val.vi.txt")
test_src, test_trg = os.path.join(DATA_DIR, "test_noisy.en.txt"), os.path.join(DATA_DIR, "test.vi.txt")

baseline_train_pairs = read_langs(train_src, train_trg, max_samples=MAX_TRAIN_SAMPLES)
baseline_val_pairs = read_langs(val_src, val_trg)
baseline_test_pairs = read_langs(test_src, test_trg)

baseline_src_vocab, baseline_trg_vocab = Vocabulary(), Vocabulary()
for s, t in baseline_train_pairs:
    baseline_src_vocab.add_sentence(s)
    baseline_trg_vocab.add_sentence(t)
print(f"Baseline source vocab: {baseline_src_vocab.num_words:,}  target vocab: {baseline_trg_vocab.num_words:,}")

baseline_train_ds = BaselineDataset(baseline_train_pairs, baseline_src_vocab, baseline_trg_vocab, MAX_LEN)
baseline_val_ds = BaselineDataset(baseline_val_pairs, baseline_src_vocab, baseline_trg_vocab, MAX_LEN)
baseline_test_ds = BaselineDataset(baseline_test_pairs, baseline_src_vocab, baseline_trg_vocab, MAX_LEN)
baseline_train_loader = DataLoader(baseline_train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=baseline_collate_fn)
baseline_val_loader = DataLoader(baseline_val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=baseline_collate_fn)

baseline_enc = BaselineEncoder(baseline_src_vocab.num_words, 256, 512)
baseline_dec = BaselineDecoder(baseline_trg_vocab.num_words, 256, 512)
baseline_model = BaselineSeq2Seq(baseline_enc, baseline_dec, device).to(device)
print(f"Baseline trainable parameters: {count_parameters(baseline_model):,}")

baseline_criterion = nn.CrossEntropyLoss(ignore_index=0)
baseline_optimizer = torch.optim.Adam(baseline_model.parameters(), lr=1e-3)

baseline_train_losses, baseline_val_losses = [], []
print(f"\nTraining baseline for {BASELINE_EPOCHS} epochs...")
for epoch in range(BASELINE_EPOCHS):
    baseline_model.train()
    epoch_loss = 0.0
    for src, trg in tqdm(baseline_train_loader, desc=f"baseline epoch {epoch+1}", leave=False):
        src, trg = src.to(device), trg.to(device)
        baseline_optimizer.zero_grad()
        output = baseline_model(src, trg, 0.5)
        output_dim = output.shape[-1]
        loss = baseline_criterion(output[:, 1:].reshape(-1, output_dim), trg[:, 1:].reshape(-1))
        loss.backward()
        nn.utils.clip_grad_norm_(baseline_model.parameters(), 1.0)
        baseline_optimizer.step()
        epoch_loss += loss.item()
    avg_train_loss = epoch_loss / len(baseline_train_loader)
    baseline_train_losses.append(avg_train_loss)

    baseline_model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for src, trg in baseline_val_loader:
            src, trg = src.to(device), trg.to(device)
            output = baseline_model(src, trg, 0)
            output_dim = output.shape[-1]
            val_loss += baseline_criterion(output[:, 1:].reshape(-1, output_dim), trg[:, 1:].reshape(-1)).item()
    avg_val_loss = val_loss / len(baseline_val_loader)
    baseline_val_losses.append(avg_val_loss)
    print(f"Epoch {epoch+1:02d}/{BASELINE_EPOCHS} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

In [ ]:
def baseline_translate_sentence(model, sentence, src_vocab, trg_vocab, device, max_len=50):
    model.eval()
    tokens = [src_vocab.word2index.get(t, 1) for t in sentence.split()]
    tokens = [2] + tokens + [3]
    src_tensor = torch.LongTensor(tokens).unsqueeze(0).to(device)
    with torch.no_grad():
        _, hidden = model.encoder(src_tensor)
    trg_tokens = [2]
    for _ in range(max_len):
        trg_tensor = torch.LongTensor([trg_tokens[-1]]).to(device)
        with torch.no_grad():
            output, hidden = model.decoder(trg_tensor, hidden)
        best_token = output.argmax(1).item()
        trg_tokens.append(best_token)
        if best_token == 3:
            break
    return [trg_vocab.index2word.get(idx, "<unk>") for idx in trg_tokens[1:]]

def baseline_calculate_bleu(model, pairs, src_vocab, trg_vocab, device, max_samples=200):
    targets, predictions = [], []
    for src_s, trg_s in pairs[:max_samples]:
        pred_words = baseline_translate_sentence(model, src_s, src_vocab, trg_vocab, device)
        if pred_words and pred_words[-1] == "<eos>":
            pred_words = pred_words[:-1]
        targets.append([trg_s.split()])
        predictions.append(pred_words)
    smoothing = SmoothingFunction().method4
    return corpus_bleu([t for t in targets], predictions, smoothing_function=smoothing) * 100

baseline_bleu = baseline_calculate_bleu(baseline_model, baseline_test_pairs, baseline_src_vocab, baseline_trg_vocab, device, max_samples=200)
print(f"Baseline (vanilla RNN, greedy) BLEU on noisy test set: {baseline_bleu:.2f}")

plt.figure(figsize=(8, 5))
plt.plot(baseline_train_losses, label="Train Loss", color="dodgerblue", lw=2)
plt.plot(baseline_val_losses, label="Validation Loss", color="crimson", lw=2)
plt.title("Baseline (vanilla RNN): Training and Validation Loss")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.grid(True)
plt.savefig(os.path.join(OUTPUT_DIR, "baseline_loss_curve.png"), dpi=150, bbox_inches="tight")
plt.show()

## 4. Joint BPE tokenizer

Why BPE instead of the baseline's whitespace-token `Vocabulary`: noisy
text (missing spaces, garbage strings, typos) explodes a whitespace
vocabulary with one-off "words" seen once and never learned well -- a
subword vocabulary caps vocab size, represents unseen/garbled tokens as
combinations of known subword pieces, and is inherently more robust to
spelling noise. A single vocabulary is trained jointly over the
concatenation of cleaned English + Vietnamese text (only on the training
split, to avoid leaking val/test statistics), and shared by the encoder
input, decoder input, and the model's output projection.

In [ ]:
class SubwordTokenizer:
    def __init__(self, vocab_size: int = 8000):
        self.vocab_size = vocab_size
        self.tk = Tokenizer(BPE(unk_token=UNK))
        self.tk.normalizer = NormSequence([NFKC(), Lowercase()])
        self.tk.pre_tokenizer = Whitespace()

    def train(self, lines):
        trainer = BpeTrainer(vocab_size=self.vocab_size, special_tokens=SPECIAL_TOKENS, min_frequency=2)
        self.tk.train_from_iterator(lines, trainer=trainer)

    def encode_ids(self, text: str):
        return self.tk.encode(text).ids

    def encode_with_specials(self, text: str):
        return [SOS_ID] + self.encode_ids(text) + [EOS_ID]

    def decode_ids(self, ids, skip_specials=True):
        if skip_specials:
            special_ids = (PAD_ID, SOS_ID, EOS_ID, SEP_ID, TOEN_ID, TOVI_ID)
            ids = [i for i in ids if i not in special_ids]
        return self.tk.decode(ids)

    @property
    def vocab_size_actual(self):
        return self.tk.get_vocab_size()

    def save(self, path):
        self.tk.save(path)

    @classmethod
    def load(cls, path):
        obj = cls()
        obj.tk = Tokenizer.from_file(path)
        obj.vocab_size = obj.tk.get_vocab_size()
        return obj

def build_or_load_tokenizer(cleaned_lines, save_path, vocab_size=8000):
    if os.path.exists(save_path):
        return SubwordTokenizer.load(save_path)
    tok = SubwordTokenizer(vocab_size=vocab_size)
    tok.train(cleaned_lines)
    os.makedirs(os.path.dirname(save_path) or ".", exist_ok=True)
    tok.save(save_path)
    return tok

## 5. Data pipeline for the encoder-decoder model

Per example:
```
encoder_ids  = <sos> <toXX> a_tokens... <eos>
decoder_in   = <sos> b_tokens...              (teacher-forcing input)
decoder_tgt  = b_tokens... <eos>              (shifted-by-one loss target)
```
where `(a, b) = (src, trg)` normally (`en2vi`, the real task), or
`(trg, src)` for a fraction (`p_reverse`) of training examples packed in
reverse (`vi2en`) -- this is what makes the model bidirectional, used
later by the reverse-model scoring term in the rerank pipeline. Source-noise
augmentation (§1b) is applied to the training split only.

In [ ]:
VOCAB_SIZE = 8000
P_REVERSE = 0.3
AUGMENT_NOISE = True

def read_parallel(src_path, trg_path, max_samples=None):
    with open(src_path, "r", encoding="utf-8") as f:
        src_lines = [l.strip() for l in f.readlines()]
    with open(trg_path, "r", encoding="utf-8") as f:
        trg_lines = [l.strip() for l in f.readlines()]
    assert len(src_lines) == len(trg_lines)
    pairs = list(zip(src_lines, trg_lines))
    return pairs[:max_samples] if max_samples is not None else pairs

def encode_pair(src_text, trg_text, tok, max_len, direction="en2vi"):
    tag_id = TOVI_ID if direction == "en2vi" else TOEN_ID
    a_text, b_text = (src_text, trg_text) if direction == "en2vi" else (trg_text, src_text)
    enc_budget, dec_budget = max_len - 3, max_len - 2
    a_ids = tok.encode_ids(a_text)[:enc_budget]
    b_ids = tok.encode_ids(b_text)[:dec_budget]
    encoder_ids = [SOS_ID, tag_id] + a_ids + [EOS_ID]
    decoder_input = [SOS_ID] + b_ids
    decoder_target = b_ids + [EOS_ID]
    return encoder_ids, decoder_input, decoder_target

class TranslationDataset(Dataset):
    def __init__(self, cleaned_pairs, tok, max_len=128, p_reverse=0.0, augment=False, seed=42):
        self.pairs, self.tok, self.max_len = cleaned_pairs, tok, max_len
        self.p_reverse, self.augment = p_reverse, augment
        self.rng = random.Random(seed)

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        src_s, trg_s = self.pairs[idx]
        if self.augment:
            src_s = augment_noise(src_s, self.rng)
        direction = "vi2en" if (self.p_reverse > 0 and self.rng.random() < self.p_reverse) else "en2vi"
        enc_ids, dec_in, dec_tgt = encode_pair(src_s, trg_s, self.tok, self.max_len, direction=direction)
        return torch.tensor(enc_ids), torch.tensor(dec_in), torch.tensor(dec_tgt)

def collate_fn(batch):
    enc_list, dec_in_list, dec_tgt_list = zip(*batch)
    enc_padded = nn.utils.rnn.pad_sequence(enc_list, batch_first=True, padding_value=PAD_ID)
    dec_in_padded = nn.utils.rnn.pad_sequence(dec_in_list, batch_first=True, padding_value=PAD_ID)
    dec_tgt_padded = nn.utils.rnn.pad_sequence(dec_tgt_list, batch_first=True, padding_value=PAD_ID)
    return enc_padded, dec_in_padded, dec_tgt_padded

# Prefer the fully-denoised copy from step 2; fall back to raw + clean_text.
def _resolve_split(split, raw_src_name, raw_trg_name, max_samples=None):
    clean_src = os.path.join(CLEAN_DIR, f"{split}_clean.en.txt")
    clean_trg = os.path.join(CLEAN_DIR, f"{split}_clean.vi.txt")
    if os.path.exists(clean_src) and os.path.exists(clean_trg):
        return read_parallel(clean_src, clean_trg, max_samples=max_samples)
    raw_pairs = read_parallel(os.path.join(DATA_DIR, raw_src_name), os.path.join(DATA_DIR, raw_trg_name), max_samples=max_samples)
    return [(clean_text(s), clean_text(t)) for s, t in raw_pairs]

train_pairs_clean = _resolve_split("train", "train_noisy.en.txt", "train.vi.txt", max_samples=MAX_TRAIN_SAMPLES)
val_pairs_clean = _resolve_split("val", "val_noisy.en.txt", "val.vi.txt")
test_pairs_clean = _resolve_split("test", "test_noisy.en.txt", "test.vi.txt")
print(f"train/val/test pairs: {len(train_pairs_clean)}/{len(val_pairs_clean)}/{len(test_pairs_clean)}")

os.makedirs(TOK_DIR, exist_ok=True)
joint_lines = (s for pair in train_pairs_clean for s in pair)
tok = build_or_load_tokenizer(joint_lines, os.path.join(TOK_DIR, "joint_bpe.json"), vocab_size=VOCAB_SIZE)
print(f"Joint BPE vocab size: {tok.vocab_size_actual}")

train_ds = TranslationDataset(train_pairs_clean, tok, max_len=MAX_LEN, p_reverse=P_REVERSE, augment=AUGMENT_NOISE)
val_ds = TranslationDataset(val_pairs_clean, tok, max_len=MAX_LEN)
test_ds = TranslationDataset(test_pairs_clean, tok, max_len=MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

enc, dec_in, dec_tgt = next(iter(train_loader))
print("encoder batch:", enc.shape, "decoder batch:", dec_in.shape)

print("\n--- Sample pairs ---")
for i in random.sample(range(len(train_pairs_clean)), 3):
    print("SRC:", train_pairs_clean[i][0])
    print("TRG:", train_pairs_clean[i][1])
    print()

## 6. Model: tiny Transformer encoder-decoder

| Component | Value |
|---|---|
| Encoder layers | 3 |
| Decoder layers | 3 |
| Model dimension | 128 |
| Attention heads | 4 |
| FFN dimension | 512 |
| Vocabulary | 8,000 joint BPE tokens |
| Embeddings | shared across encoder input, decoder input, and output projection |
| Position encoding | sinusoidal (0 extra parameters) |
| Dropout | 0.1 |

Bidirectional self-attention in the encoder over the (denoised) noisy
source; causal self-attention + cross-attention into the encoder output in
the decoder. The `DecoderLayer` is custom (rather than
`nn.TransformerDecoderLayer`) so cross-attention weights can be extracted
for the coverage-penalty / attention-map analysis. A leading direction tag
(`<toen>`/`<tovi>`) on the encoder input makes the same encoder+decoder
pair bidirectional at zero extra parameters.

**Parameter budget** (joint vocab=8000): shared embedding ~1.02M + 3
encoder layers ~0.59M + 3 decoder layers ~0.79M + biases/norms ~0.03M
&approx; **2.4M** -- clears both the 5,000,000 hard budget and the
2,500,000 bonus threshold.

In [ ]:
def _banned_next_tokens(seq, n):
    """No-repeat-n-gram blocking: ban whatever token previously followed
    the current (n-1)-token suffix, to prevent the decoder looping on a
    short phrase."""
    if n is None or n <= 0 or len(seq) < n - 1:
        return ()
    prefix = tuple(seq[-(n - 1):]) if n > 1 else ()
    banned = set()
    for i in range(len(seq) - n + 1):
        if tuple(seq[i:i + n - 1]) == prefix:
            banned.add(seq[i + n - 1])
    return banned

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe[:, : x.size(1)]

class DecoderLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.ln2 = nn.LayerNorm(d_model)
        self.cross_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.ln3 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, dim_feedforward), nn.GELU(),
                                  nn.Dropout(dropout), nn.Linear(dim_feedforward, d_model))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, memory, causal_mask, tgt_key_padding_mask, memory_key_padding_mask):
        h = self.ln1(x)
        sa_out, _ = self.self_attn(h, h, h, attn_mask=causal_mask, key_padding_mask=tgt_key_padding_mask, need_weights=False)
        x = x + self.dropout(sa_out)
        h2 = self.ln2(x)
        ca_out, ca_w = self.cross_attn(h2, memory, memory, key_padding_mask=memory_key_padding_mask,
                                         need_weights=True, average_attn_weights=True)
        x = x + self.dropout(ca_out)
        x = x + self.dropout(self.ff(self.ln3(x)))
        return x, ca_w  # ca_w: [batch, tgt_len, src_len]

class Seq2SeqTransformer(nn.Module):
    def __init__(self, vocab_size, device, d_model=128, nhead=4, num_encoder_layers=3,
                 num_decoder_layers=3, dim_feedforward=512, dropout=0.1, max_len=128,
                 pad_id=PAD_ID, sos_id=SOS_ID, eos_id=EOS_ID):
        super().__init__()
        self.device, self.d_model, self.max_len = device, d_model, max_len
        self.pad_id, self.sos_id, self.eos_id = pad_id, sos_id, eos_id
        self.tok_emb = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.pos_enc = PositionalEncoding(d_model, max_len=max_len)
        self.emb_dropout = nn.Dropout(dropout)
        enc_layer = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward, dropout, batch_first=True, activation="relu")
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_encoder_layers)
        self.decoder_layers = nn.ModuleList([DecoderLayer(d_model, nhead, dim_feedforward, dropout) for _ in range(num_decoder_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.output_bias = nn.Parameter(torch.zeros(vocab_size))

    def _embed(self, ids):
        return self.emb_dropout(self.pos_enc(self.tok_emb(ids) * math.sqrt(self.d_model)))

    def _output_proj(self, x):
        return F.linear(x, self.tok_emb.weight, self.output_bias)

    @staticmethod
    def _causal_mask(sz, device):
        return torch.triu(torch.full((sz, sz), float("-inf"), device=device), diagonal=1)

    def encode(self, enc_ids):
        memory_kpm = enc_ids == self.pad_id
        x = self._embed(enc_ids)
        memory = self.encoder(x, src_key_padding_mask=memory_kpm)
        return memory, memory_kpm

    def decode(self, dec_ids, memory, memory_kpm):
        tgt_kpm = dec_ids == self.pad_id
        causal_mask = self._causal_mask(dec_ids.size(1), dec_ids.device)
        x = self._embed(dec_ids)
        cross_attn = None
        for layer in self.decoder_layers:
            x, cross_attn = layer(x, memory, causal_mask, tgt_kpm, memory_kpm)
        x = self.ln_f(x)
        return self._output_proj(x), cross_attn

    def forward(self, enc_ids, dec_ids):
        memory, memory_kpm = self.encode(enc_ids)
        return self.decode(dec_ids, memory, memory_kpm)

    @torch.no_grad()
    def greedy_generate(self, enc_ids, max_new_tokens=60, no_repeat_ngram_size=3, min_length=1):
        assert enc_ids.shape[0] == 1
        self.eval()
        memory, memory_kpm = self.encode(enc_ids)
        dec_ids = torch.tensor([[self.sos_id]], dtype=torch.long, device=enc_ids.device)
        last_attn = None
        for _ in range(max_new_tokens):
            if dec_ids.size(1) >= self.max_len:
                break
            logits, attn = self.decode(dec_ids, memory, memory_kpm)
            last_attn = attn
            step_logits = logits[0, -1].clone()
            if dec_ids.size(1) - 1 < min_length:
                step_logits[self.eos_id] = float("-inf")
            for banned in _banned_next_tokens(dec_ids[0].tolist(), no_repeat_ngram_size):
                step_logits[banned] = float("-inf")
            next_tok = step_logits.argmax().view(1, 1)
            dec_ids = torch.cat([dec_ids, next_tok], dim=1)
            if next_tok.item() == self.eos_id:
                break
        return dec_ids[0].tolist(), last_attn

    @torch.no_grad()
    def beam_search_generate(self, enc_ids, max_new_tokens=60, beam_width=5, length_penalty=0.7,
                               no_repeat_ngram_size=3, min_length=1):
        candidates = self.beam_search_candidates(enc_ids, max_new_tokens=max_new_tokens, beam_width=beam_width,
                                                    num_return=1, length_penalty=length_penalty,
                                                    no_repeat_ngram_size=no_repeat_ngram_size, min_length=min_length)
        return candidates[0]["seq"]

    @torch.no_grad()
    def beam_search_candidates(self, enc_ids, max_new_tokens=60, beam_width=10, num_return=10,
                                 length_penalty=0.7, no_repeat_ngram_size=3, min_length=1):
        assert enc_ids.shape[0] == 1
        self.eval()
        memory, memory_kpm = self.encode(enc_ids)
        src_len = memory.shape[1]

        def norm_score(seq, score):
            return score / max(1, len(seq) - 1) ** length_penalty

        beams = [([self.sos_id], 0.0, False, [0.0] * src_len)]
        finished_hyps = []
        for _ in range(max_new_tokens):
            candidates, any_active = [], False
            for seq, score, finished, cov in beams:
                if finished or len(seq) >= self.max_len:
                    if finished:
                        finished_hyps.append((seq, score, cov))
                    continue
                any_active = True
                dec_ids = torch.tensor([seq], dtype=torch.long, device=enc_ids.device)
                logits, attn = self.decode(dec_ids, memory, memory_kpm)
                step_attn = attn[0, -1, :].tolist()
                log_probs = F.log_softmax(logits[0, -1], dim=-1).clone()
                if len(seq) - 1 < min_length:
                    log_probs[self.eos_id] = float("-inf")
                for banned in _banned_next_tokens(seq, no_repeat_ngram_size):
                    log_probs[banned] = float("-inf")
                topk = min(beam_width, log_probs.size(-1))
                topk_log_probs, topk_ids = log_probs.topk(topk)
                for k in range(topk):
                    tok_id = topk_ids[k].item()
                    new_cov = [c + a for c, a in zip(cov, step_attn)]
                    candidates.append((seq + [tok_id], score + topk_log_probs[k].item(), tok_id == self.eos_id, new_cov))
            if not any_active:
                break
            candidates.sort(key=lambda c: norm_score(c[0], c[1]), reverse=True)
            beams = candidates[:beam_width]
            if len(finished_hyps) >= num_return and all(b[2] for b in beams):
                break
        already = {tuple(s) for s, _, _ in finished_hyps}
        for seq, score, finished, cov in beams:
            if tuple(seq) not in already:
                finished_hyps.append((seq, score, cov))
        finished_hyps.sort(key=lambda c: norm_score(c[0], c[1]), reverse=True)
        return [{"seq": seq, "logprob": score, "coverage": cov} for seq, score, cov in finished_hyps[:num_return]]

    @torch.no_grad()
    def sample_generate(self, enc_ids, max_new_tokens=60, temperature=0.7, top_k=20,
                          no_repeat_ngram_size=3, min_length=1):
        assert enc_ids.shape[0] == 1
        self.eval()
        memory, memory_kpm = self.encode(enc_ids)
        dec_ids = torch.tensor([[self.sos_id]], dtype=torch.long, device=enc_ids.device)
        total_logprob = 0.0
        for _ in range(max_new_tokens):
            if dec_ids.size(1) >= self.max_len:
                break
            logits, _ = self.decode(dec_ids, memory, memory_kpm)
            step_logits = logits[0, -1] / max(temperature, 1e-5)
            if dec_ids.size(1) - 1 < min_length:
                step_logits[self.eos_id] = float("-inf")
            for banned in _banned_next_tokens(dec_ids[0].tolist(), no_repeat_ngram_size):
                step_logits[banned] = float("-inf")
            if top_k and top_k > 0:
                v, _ = step_logits.topk(min(top_k, step_logits.size(-1)))
                step_logits = step_logits.clone()
                step_logits[step_logits < v[-1]] = float("-inf")
            probs = F.softmax(step_logits, dim=-1)
            next_tok = torch.multinomial(probs, 1)
            total_logprob += torch.log(probs[next_tok] + 1e-12).item()
            dec_ids = torch.cat([dec_ids, next_tok.view(1, 1)], dim=1)
            if next_tok.item() == self.eos_id:
                break
        return {"seq": dec_ids[0].tolist(), "logprob": total_logprob}

    @torch.no_grad()
    def score_sequence(self, enc_ids, dec_full_ids):
        dec_full_ids = dec_full_ids if dec_full_ids.dim() == 2 else dec_full_ids.unsqueeze(0)
        memory, memory_kpm = self.encode(enc_ids)
        logits, _ = self.decode(dec_full_ids[:, :-1], memory, memory_kpm)
        log_probs = F.log_softmax(logits, dim=-1)
        targets = dec_full_ids[:, 1:]
        token_logprobs = log_probs.gather(-1, targets.unsqueeze(-1)).squeeze(-1)
        return token_logprobs.sum().item(), token_logprobs[0].tolist()

    @torch.no_grad()
    def coverage_vector(self, enc_ids, dec_full_ids):
        dec_full_ids = dec_full_ids if dec_full_ids.dim() == 2 else dec_full_ids.unsqueeze(0)
        memory, memory_kpm = self.encode(enc_ids)
        _, cross_attn = self.decode(dec_full_ids[:, :-1], memory, memory_kpm)
        if cross_attn is None:
            return None
        return cross_attn[0].sum(dim=0).tolist()

def build_model(vocab_size, device, d_model=128, nhead=4, num_encoder_layers=3,
                 num_decoder_layers=3, dim_feedforward=512, dropout=0.1, max_len=128):
    return Seq2SeqTransformer(vocab_size, device, d_model=d_model, nhead=nhead,
                                num_encoder_layers=num_encoder_layers, num_decoder_layers=num_decoder_layers,
                                dim_feedforward=dim_feedforward, dropout=dropout, max_len=max_len).to(device)

D_MODEL, NHEAD, NUM_ENCODER_LAYERS, NUM_DECODER_LAYERS, D_FF, DROPOUT = 128, 4, 3, 3, 512, 0.1
PARAM_BUDGET, BONUS_PARAM_BUDGET = 5_000_000, 2_500_000

model = build_model(tok.vocab_size_actual, device, d_model=D_MODEL, nhead=NHEAD,
                      num_encoder_layers=NUM_ENCODER_LAYERS, num_decoder_layers=NUM_DECODER_LAYERS,
                      dim_feedforward=D_FF, dropout=DROPOUT, max_len=MAX_LEN)

n_params = count_parameters(model)
print(f"Trainable parameters: {n_params:,} (hard budget: {PARAM_BUDGET:,})")
assert n_params <= PARAM_BUDGET, f"Model has {n_params:,} params, exceeding the {PARAM_BUDGET:,} budget!"
if n_params <= BONUS_PARAM_BUDGET:
    margin = BONUS_PARAM_BUDGET - n_params
    print(f"Clears the bonus threshold (<={BONUS_PARAM_BUDGET:,} params) with {margin:,} params ({100*margin/BONUS_PARAM_BUDGET:.1f}%) to spare.")
else:
    print(f"NOTE: exceeds the {BONUS_PARAM_BUDGET:,} bonus threshold (still within the {PARAM_BUDGET:,} hard budget).")

## 7. Train

Standard cross-entropy over the decoder target (`<pad>` ignored), with
label smoothing (useful since noisy sources make some target tokens
genuinely ambiguous), gradient clipping, and `ReduceLROnPlateau`. Also
keeps a rolling window of the last-K epoch checkpoints for optional
weight averaging (§10).

In [ ]:
LR = 3e-4
LABEL_SMOOTHING = 0.1
SAVE_LAST_K = 3

criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID, label_smoothing=LABEL_SMOOTHING)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, betas=(0.9, 0.98), eps=1e-9)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

def run_epoch(model, loader, optimizer, criterion, device, train=True):
    model.train() if train else model.eval()
    total_loss = 0.0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for enc_ids, dec_in, dec_tgt in loader:
            enc_ids, dec_in, dec_tgt = enc_ids.to(device), dec_in.to(device), dec_tgt.to(device)
            if train:
                optimizer.zero_grad()
            logits, _ = model(enc_ids, dec_in)
            loss = criterion(logits.reshape(-1, logits.shape[-1]), dec_tgt.reshape(-1))
            if train:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            total_loss += loss.item()
    return total_loss / len(loader)

ckpts_dir = os.path.join(OUTPUT_DIR, "ckpts")
os.makedirs(ckpts_dir, exist_ok=True)
ckpt_path = os.path.join(OUTPUT_DIR, "checkpoint.pt")

train_losses, val_losses = [], []
best_val_loss = float("inf")
saved_ckpts = []

print(f"Training for {EPOCHS} epochs...")
for epoch in range(EPOCHS):
    start = time.time()
    train_loss = run_epoch(model, train_loader, optimizer, criterion, device, train=True)
    val_loss = run_epoch(model, val_loader, optimizer, criterion, device, train=False)
    scheduler.step(val_loss)
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    elapsed = time.time() - start

    ckpt_payload = {
        "model_state_dict": model.state_dict(),
        "config": dict(d_model=D_MODEL, nhead=NHEAD, num_encoder_layers=NUM_ENCODER_LAYERS,
                        num_decoder_layers=NUM_DECODER_LAYERS, d_ff=D_FF, dropout=DROPOUT, max_len=MAX_LEN),
        "vocab_size": tok.vocab_size_actual,
        "train_losses": train_losses, "val_losses": val_losses,
    }
    is_best = val_loss < best_val_loss
    if is_best:
        best_val_loss = val_loss
        torch.save(ckpt_payload, ckpt_path)

    epoch_ckpt_path = os.path.join(ckpts_dir, f"epoch_{epoch+1:03d}.pt")
    torch.save(ckpt_payload, epoch_ckpt_path)
    saved_ckpts.append(epoch_ckpt_path)
    if len(saved_ckpts) > SAVE_LAST_K:
        stale = saved_ckpts.pop(0)
        if os.path.exists(stale):
            os.remove(stale)

    print(f"Epoch {epoch+1:02d}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | {elapsed:.1f}s | best: {is_best}")

plt.figure(figsize=(8, 5))
plt.plot(train_losses, label="Train Loss", color="dodgerblue", lw=2)
plt.plot(val_losses, label="Validation Loss", color="crimson", lw=2)
plt.title("Model Training and Validation Cross-Entropy Loss")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.grid(True)
plt.savefig(os.path.join(OUTPUT_DIR, "loss_curve.png"), dpi=150, bbox_inches="tight")
plt.show()
print(f"Best checkpoint (val_loss={best_val_loss:.4f}) saved to {ckpt_path}")

## 7b. (Optional) Checkpoint averaging

Averages the weights of the last `SAVE_LAST_K` epoch checkpoints
(SWA-style) -- a free way to often pick up a bit of BLEU without changing
the parameter count. Run this cell, then re-run the evaluation cells below
with `model.load_state_dict(avg_state)` to compare.

In [ ]:
import glob

def average_checkpoints(ckpt_paths):
    payloads = [torch.load(p, map_location="cpu") for p in ckpt_paths]
    state_dicts = [p["model_state_dict"] for p in payloads]
    avg_state = {}
    for key in state_dicts[0].keys():
        tensors = [sd[key].float() for sd in state_dicts]
        avg_state[key] = torch.stack(tensors, dim=0).mean(dim=0).to(state_dicts[0][key].dtype)
    return avg_state

ckpt_paths = sorted(glob.glob(os.path.join(ckpts_dir, "epoch_*.pt")))
if len(ckpt_paths) >= 2:
    avg_state = average_checkpoints(ckpt_paths)
    torch.save({"model_state_dict": avg_state}, os.path.join(OUTPUT_DIR, "checkpoint_avg.pt"))
    print(f"Averaged {len(ckpt_paths)} checkpoints -> {os.path.join(OUTPUT_DIR, 'checkpoint_avg.pt')}")
    # Uncomment to actually use the averaged weights for the evaluation below:
    # model.load_state_dict(avg_state)
else:
    print(f"Only {len(ckpt_paths)} checkpoint(s) available -- need >=2 to average (raise SAVE_LAST_K / EPOCHS).")

## 8. The generate-many-then-rerank pipeline

Beyond plain greedy/beam decoding:

1. **Generate 10-30 candidates**: a k-best beam search pool plus several
   low-temperature top-k sampling draws (beam search alone tends toward
   near-duplicate high-probability hypotheses; sampling fills in
   genuinely different ones). The encoder runs once per sentence and its
   output is reused across every candidate.
2. **Filter malformed candidates**: missing `<eos>` within budget, leaked
   special tokens, degenerate/empty output, excessive repeated n-grams.
3. **No-repeat n-gram blocking** and **min-length control** are applied
   *during* generation itself, not just as a post-hoc filter.
4. **Rerank the survivors** with a translation-oriented score:

$$
S(y|x) = \frac{\log P(y|x)}{|y|^{\alpha}}
          + \lambda_{cov} \cdot C(x,y)
          + \lambda_{rev} \cdot \log P(x|y)
          - \lambda_{rep} \cdot R(y)
$$

- length-normalized `log P(y|x)` (teacher-forced score under the model)
- `C(x,y)`: GNMT-style coverage penalty from the decoder's own
  cross-attention into the encoder -- rewards attending to every source
  token at least once
- `log P(x|y)`: reverse-direction score from the SAME model (trained
  bidirectionally, §5) -- does translating the candidate back reconstruct
  the noisy source reasonably well?
- `R(y)`: repeated-trigram fraction

5. **Optional MBR reranking**: instead of the top `S(y|x)` candidate,
   pick among the top few survivors whichever has the highest average
   **chrF** similarity to the others -- a self-consistency signal.

In [ ]:
def _char_ngrams(s, n):
    s = s.replace(" ", "")
    return [s[i:i + n] for i in range(len(s) - n + 1)]

def chrf_score(hyp, ref, max_n=6, beta=2.0):
    if not hyp or not ref:
        return 0.0
    precisions, recalls = [], []
    for n in range(1, max_n + 1):
        h_ngrams, r_ngrams = Counter(_char_ngrams(hyp, n)), Counter(_char_ngrams(ref, n))
        match = sum((h_ngrams & r_ngrams).values())
        h_total, r_total = sum(h_ngrams.values()), sum(r_ngrams.values())
        precisions.append(match / h_total if h_total else 0.0)
        recalls.append(match / r_total if r_total else 0.0)
    P, R = sum(precisions) / max_n, sum(recalls) / max_n
    return 0.0 if P + R == 0 else (1 + beta ** 2) * P * R / (beta ** 2 * P + R)

def mbr_select(texts):
    n = len(texts)
    if n == 0:
        return None, []
    if n == 1:
        return 0, [1.0]
    scores = [sum(chrf_score(texts[i], texts[j]) for j in range(n) if j != i) / (n - 1) for i in range(n)]
    return max(range(n), key=lambda i: scores[i]), scores

def repetition_penalty(tokens, n=3):
    if len(tokens) < n:
        return 0.0
    ngrams = [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]
    counts = Counter(ngrams)
    return sum(c - 1 for c in counts.values() if c > 1) / max(1, len(ngrams))

def filter_malformed(candidates, max_new_tokens, special_ids, min_gen_len=1, max_rep=0.3):
    kept, reasons = [], []
    for cand in candidates:
        gen = cand["seq"][1:]
        if EOS_ID not in gen:
            reasons.append("missing_eos"); continue
        body = gen[: gen.index(EOS_ID)]
        if len(body) < min_gen_len:
            reasons.append("too_short"); continue
        if len(gen) > max_new_tokens:
            reasons.append("excessive_length"); continue
        if any(t in special_ids for t in body):
            reasons.append("leaked_special_token"); continue
        if repetition_penalty(body, n=3) > max_rep:
            reasons.append("excessive_repetition"); continue
        cand = dict(cand); cand["body"] = body
        kept.append(cand)
    return kept, reasons

def score_candidates(model, enc_ids_tensor, src_ids, src_span, candidates,
                       alpha=0.9, lambda_cov=0.3, lambda_rev=0.2, lambda_rep=0.5):
    dev = next(model.parameters()).device
    out = []
    for cand in candidates:
        body = cand["body"]
        dec_full = torch.tensor([[SOS_ID] + body + [EOS_ID]], dtype=torch.long, device=dev)
        fwd_logprob, _ = model.score_sequence(enc_ids_tensor, dec_full)
        length_norm = fwd_logprob / (len(body) + 1) ** alpha

        cov_term, coverage = 0.0, None
        if lambda_cov and src_span is not None:
            full_cov = model.coverage_vector(enc_ids_tensor, dec_full)
            if full_cov:
                coverage = full_cov[src_span[0]:src_span[1]]
                cov_term = sum(math.log(min(c, 1.0) + 1e-6) for c in coverage)

        rev_logprob = 0.0
        if lambda_rev and src_ids:
            rev_enc = torch.tensor([[SOS_ID, TOEN_ID] + body + [EOS_ID]], dtype=torch.long, device=dev)
            rev_dec = torch.tensor([[SOS_ID] + src_ids + [EOS_ID]], dtype=torch.long, device=dev)
            if rev_enc.shape[1] <= model.max_len and rev_dec.shape[1] <= model.max_len:
                raw_rev_logprob, _ = model.score_sequence(rev_enc, rev_dec)
                rev_logprob = raw_rev_logprob / (len(src_ids) + 1)

        rep = repetition_penalty(body, n=3)
        score = length_norm + lambda_cov * cov_term + lambda_rev * rev_logprob - lambda_rep * rep
        out.append({**cand, "score": score, "logprob_fwd": fwd_logprob, "logprob_fwd_norm": length_norm,
                     "logprob_rev_norm": rev_logprob, "coverage": coverage, "repetition": rep})
    out.sort(key=lambda c: c["score"], reverse=True)
    return out

def generate_and_rerank(model, tok, src_text, max_len, device,
                          n_beam=15, n_sample=10, beam_width=10, temperature=0.7, top_k=20,
                          max_new_tokens=60, no_repeat_ngram_size=3, min_length=1,
                          alpha=0.9, lambda_cov=0.3, lambda_rev=0.2, lambda_rep=0.5,
                          use_mbr=False, mbr_pool=5):
    encoder_ids, _, _ = encode_pair(src_text, "", tok, max_len, direction="en2vi")
    enc_ids_tensor = torch.tensor([encoder_ids], dtype=torch.long, device=device)
    src_ids = encoder_ids[2:-1]
    src_span = (2, len(encoder_ids) - 1)

    raw_candidates = []
    beam_cands = model.beam_search_candidates(enc_ids_tensor, max_new_tokens=max_new_tokens, beam_width=beam_width,
                                                 num_return=n_beam, no_repeat_ngram_size=no_repeat_ngram_size, min_length=min_length)
    raw_candidates.extend({"seq": c["seq"]} for c in beam_cands)
    for _ in range(n_sample):
        s = model.sample_generate(enc_ids_tensor, max_new_tokens=max_new_tokens, temperature=temperature, top_k=top_k,
                                    no_repeat_ngram_size=no_repeat_ngram_size, min_length=min_length)
        raw_candidates.append({"seq": s["seq"]})

    special_ids = set(range(len(SPECIAL_TOKENS)))
    kept, dropped_reasons = filter_malformed(raw_candidates, max_new_tokens=max_new_tokens, special_ids=special_ids)
    if not kept:
        greedy_seq, _ = model.greedy_generate(enc_ids_tensor, max_new_tokens=max_new_tokens,
                                                 no_repeat_ngram_size=no_repeat_ngram_size, min_length=min_length)
        gen = greedy_seq[1:]
        body = gen[: gen.index(EOS_ID)] if EOS_ID in gen else gen
        kept = [{"seq": greedy_seq, "body": body}]

    seen, deduped = set(), []
    for c in kept:
        key = tuple(c["body"])
        if key in seen:
            continue
        seen.add(key); deduped.append(c)

    scored = score_candidates(model, enc_ids_tensor, src_ids, src_span, deduped,
                                alpha=alpha, lambda_cov=lambda_cov, lambda_rev=lambda_rev, lambda_rep=lambda_rep)

    if use_mbr and len(scored) > 1:
        pool = scored[:mbr_pool]
        texts = [tok.decode_ids(c["body"], skip_specials=True) for c in pool]
        idx, mbr_scores = mbr_select(texts)
        for c, s in zip(pool, mbr_scores):
            c["mbr_score"] = s
        best = pool[idx]
    else:
        best = scored[0]

    best_text = tok.decode_ids(best["body"], skip_specials=True)
    return best_text, best, scored, dropped_reasons

## 9. Quantitative evaluation: BLEU for greedy / beam search / rerank

Evaluates the tiny Transformer model's BLEU on the noisy test set under
three decoding strategies, so they can be compared directly in the report.
The rerank pipeline is much more expensive per sentence (10-30 forward
trajectories vs. 1-5), so it's evaluated on a subset by default --
raise `RERANK_EVAL_SAMPLES` (or set to `None`) for the full test set in
the final report run.

In [ ]:
def ids_to_words(tok, ids):
    return tok.decode_ids(ids, skip_specials=True).split()

@torch.no_grad()
def evaluate_bleu(model, loader, tok, device, generate_fn, max_samples=None, max_new_tokens=60):
    model.eval()
    refs, hyps, n_seen = [], [], 0
    for enc_ids, _dec_in, dec_tgt in tqdm(loader, desc="Evaluating"):
        for i in range(enc_ids.shape[0]):
            if max_samples is not None and n_seen >= max_samples:
                break
            enc = enc_ids[i : i + 1].to(device)
            dec_seq = generate_fn(model, enc, max_new_tokens=max_new_tokens)
            gen_ids = dec_seq[1:]
            if EOS_ID in gen_ids:
                gen_ids = gen_ids[: gen_ids.index(EOS_ID)]
            ref_ids = [t for t in dec_tgt[i].tolist() if t != PAD_ID]
            if ref_ids and ref_ids[-1] == EOS_ID:
                ref_ids = ref_ids[:-1]
            hyps.append(ids_to_words(tok, gen_ids))
            refs.append([ids_to_words(tok, ref_ids)])
            n_seen += 1
        if max_samples is not None and n_seen >= max_samples:
            break
    smoothing = SmoothingFunction().method4
    return corpus_bleu(refs, hyps, smoothing_function=smoothing) * 100, refs, hyps

def greedy_generate_fn(model, enc_ids, max_new_tokens=60):
    seq, _ = model.greedy_generate(enc_ids, max_new_tokens=max_new_tokens)
    return seq

def beam_generate_fn(model, enc_ids, max_new_tokens=60, beam_width=5):
    return model.beam_search_generate(enc_ids, max_new_tokens=max_new_tokens, beam_width=beam_width)

@torch.no_grad()
def evaluate_bleu_rerank(model, tok, pairs, device, max_len, max_samples=None, **rerank_kwargs):
    model.eval()
    refs, hyps = [], []
    subset = pairs[:max_samples] if max_samples is not None else pairs
    for src, trg in tqdm(subset, desc="Reranked eval"):
        best_text, _, _, _ = generate_and_rerank(model, tok, src, max_len, device, **rerank_kwargs)
        hyps.append(best_text.split())
        refs.append([trg.split()])
    smoothing = SmoothingFunction().method4
    return corpus_bleu(refs, hyps, smoothing_function=smoothing) * 100

BEAM_WIDTH = 5
MAX_EVAL_SAMPLES = None          # None = full test set for greedy/beam
RERANK_EVAL_SAMPLES = 50         # keep small -- rerank is expensive per sentence
MAX_NEW_TOKENS = MAX_LEN // 2

rerank_kwargs = dict(n_beam=15, n_sample=10, beam_width=BEAM_WIDTH, temperature=0.7,
                       max_new_tokens=MAX_NEW_TOKENS, alpha=0.9, lambda_cov=0.3,
                       lambda_rev=0.2, lambda_rep=0.5, use_mbr=False)

print("=== Greedy decoding ===")
bleu_greedy, _, _ = evaluate_bleu(model, test_loader, tok, device, greedy_generate_fn,
                                     max_samples=MAX_EVAL_SAMPLES, max_new_tokens=MAX_NEW_TOKENS)
print(f"Greedy BLEU on noisy test set: {bleu_greedy:.2f}")

print("\n=== Beam search decoding ===")
bleu_beam, _, _ = evaluate_bleu(
    model, test_loader, tok, device,
    lambda m, e, max_new_tokens: beam_generate_fn(m, e, max_new_tokens=max_new_tokens, beam_width=BEAM_WIDTH),
    max_samples=MAX_EVAL_SAMPLES, max_new_tokens=MAX_NEW_TOKENS,
)
print(f"Beam search (width={BEAM_WIDTH}) BLEU on noisy test set: {bleu_beam:.2f}")

print("\n=== Generate-candidates + rerank decoding ===")
bleu_rerank = evaluate_bleu_rerank(model, tok, test_pairs_clean, device, MAX_LEN,
                                      max_samples=RERANK_EVAL_SAMPLES, **rerank_kwargs)
n_eval = RERANK_EVAL_SAMPLES or len(test_pairs_clean)
print(f"Rerank BLEU on noisy test set (n={n_eval}): {bleu_rerank:.2f}")

print("\n=== Summary ===")
print(f"Baseline (vanilla RNN, greedy) : {baseline_bleu:.2f}")
print(f"Ours -- greedy                 : {bleu_greedy:.2f}")
print(f"Ours -- beam search             : {bleu_beam:.2f}")
print(f"Ours -- generate + rerank       : {bleu_rerank:.2f}  (n={n_eval})")

## 10. Qualitative analysis: 3 noisy sentences + cross-attention maps

For each hand-picked noisy sentence: greedy, beam search, and
generate-and-rerank predictions side by side, plus the decoder's
cross-attention (which source words each generated Vietnamese token
attends to) -- look at whether attention concentrates on the real source
words rather than the noisy/garbage ones.

In [ ]:
def plot_attention(attn, src_tokens, trg_tokens, save_path=None):
    fig, ax = plt.subplots(figsize=(max(6, len(src_tokens) * 0.5), max(4, len(trg_tokens) * 0.4)))
    im = ax.imshow(attn, cmap="viridis", aspect="auto")
    ax.set_xticks(range(len(src_tokens))); ax.set_xticklabels(src_tokens, rotation=90)
    ax.set_yticks(range(len(trg_tokens))); ax.set_yticklabels(trg_tokens)
    ax.set_xlabel("Source (noisy English)"); ax.set_ylabel("Predicted (Vietnamese)")
    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150)
        print(f"Saved attention map to {save_path}")
    plt.show()

qual_sentences = [
    "i  havean   apple!!!  soooo goood lol vacx blah",
    "apples i like .",
    "the science bruh lmao vacx behind a climate headline",
]

for i, raw in enumerate(qual_sentences):
    cleaned = clean_text(raw)
    encoder_ids, _, _ = encode_pair(cleaned, "", tok, MAX_LEN)
    enc_tensor = torch.tensor([encoder_ids], dtype=torch.long, device=device)

    greedy_seq, attn = model.greedy_generate(enc_tensor, max_new_tokens=MAX_NEW_TOKENS)
    beam_seq = model.beam_search_generate(enc_tensor, max_new_tokens=MAX_NEW_TOKENS, beam_width=BEAM_WIDTH)

    def extract_gen(seq):
        gen = seq[1:]
        return gen[: gen.index(EOS_ID)] if EOS_ID in gen else gen

    rerank_text, rerank_best, _, dropped = generate_and_rerank(model, tok, cleaned, MAX_LEN, device, **rerank_kwargs)

    print(f"\n--- Example {i+1} ---")
    print(f"Raw source     : {raw}")
    print(f"Cleaned source : {cleaned}")
    print(f"Greedy pred    : {' '.join(ids_to_words(tok, extract_gen(greedy_seq)))}")
    print(f"Beam pred      : {' '.join(ids_to_words(tok, extract_gen(beam_seq)))}")
    print(f"Rerank pred    : {rerank_text}  (score={rerank_best['score']:.3f}, rep={rerank_best['repetition']:.2f}, {len(dropped)} malformed candidates dropped)")

    if attn is not None:
        trg_len = attn.shape[1]
        src_pieces = [tok.decode_ids([t], skip_specials=False) or "?" for t in encoder_ids[2:-1]]
        trg_pieces = [tok.decode_ids([t], skip_specials=False) or "?" for t in greedy_seq[1 : trg_len + 1]]
        attn_matrix = attn[0, :, 2:-1].cpu().numpy()
        plot_attention(attn_matrix, src_pieces, trg_pieces,
                         save_path=os.path.join(OUTPUT_DIR, f"attention_example_{i+1}.png"))

## 11. Error analysis: clean vs. noisy inputs

Same underlying sentence, clean vs. artificially noised, to see how much
the model's output degrades under noise.

In [ ]:
error_pairs = [
    ("i like apples .", "i  lyke aplz!!! "),
    ("the weather is nice today .", "thewether iz nyce todayyy"),
]

for clean_variant, noisy_variant in error_pairs:
    print(f"\nClean : {clean_variant}")
    print(f"Noisy : {noisy_variant}")
    for label, s in [("clean", clean_variant), ("noisy", noisy_variant)]:
        cleaned = clean_text(s)
        encoder_ids, _, _ = encode_pair(cleaned, "", tok, MAX_LEN)
        enc_tensor = torch.tensor([encoder_ids], dtype=torch.long, device=device)
        seq, _ = model.greedy_generate(enc_tensor, max_new_tokens=MAX_NEW_TOKENS)
        gen = seq[1:]
        gen = gen[: gen.index(EOS_ID)] if EOS_ID in gen else gen
        print(f"  [{label:>5}] cleaned='{cleaned}' -> {' '.join(ids_to_words(tok, gen))}")

## 12. Wrap-up

- `output/checkpoint.pt` -- best model weights + config + loss history
- `output/loss_curve.png`, `output/baseline_loss_curve.png`
- `output/attention_example_{1,2,3}.png` -- cross-attention heatmaps
- `output/preprocess_stats.json` -- denoising statistics per split
- `output/ckpts/` -- rolling checkpoints for weight averaging

Re-run §9/§10 after loading `output/checkpoint_avg.pt` (§7b) to compare
checkpoint-averaged BLEU against the single best checkpoint for the report.